# Training Jobs

## Understanding Training Jobs

Training Jobs run your training script on managed infrastructure. They handle data loading, model training, and checkpoint management. SageMaker automatically scales resources and manages failures.

## Using the Estimator API

In [ ]:
from sagemaker.estimator import Estimator
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'
bucket = session.default_bucket()

# Create estimator for custom training script
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/training-output',
    code_location=f's3://{bucket}/code',
    sagemaker_session=session
)

# Set hyperparameters
estimator.set_hyperparameters(
    epochs=10,
    batch_size=32,
    learning_rate=0.001,
    optimizer='adam'
)

# Start training
estimator.fit(
    {'training': f's3://{bucket}/train-data/'},
    job_name='training-job-2024-01-15',
    wait=True
)

## Hyperparameter Configuration

In [ ]:
from sagemaker.tensorflow import TensorFlow

# TensorFlow estimator with hyperparameters
tf_estimator = TensorFlow(
    entry_point='train.py',
    role=role,
    instance_count=1,
    instance_type='ml.p3.2xlarge',
    framework_version='2.8',
    py_version='py39',
    output_path=f's3://{bucket}/tf-output',
    sagemaker_session=session,
    hyperparameters={
        'epochs': 50,
        'batch_size': 64,
        'learning_rate': 0.001,
        'dropout': 0.5,
        'activation': 'relu'
    }
)

# Fit the model
tf_estimator.fit(
    {'training': f's3://{bucket}/train-data/'},
    job_name='tensorflow-training'
)

## Spot Training for Cost Savings

In [ ]:
from sagemaker.estimator import Estimator

# Enable spot training
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/training-output',
    sagemaker_session=session,
    use_spot_instances=True,
    max_run=3600,
    max_wait=5400
)

# Spot training can save up to 90% on compute costs
estimator.fit(
    {'training': f's3://{bucket}/train-data/'},
    job_name='spot-training-job'
)

## Distributed Training

In [ ]:
from sagemaker.pytorch import PyTorch

# Distributed training with multiple instances
pytorch_estimator = PyTorch(
    entry_point='train.py',
    role=role,
    instance_count=4,  # Multiple instances
    instance_type='ml.p3.8xlarge',
    framework_version='1.12',
    py_version='py38',
    output_path=f's3://{bucket}/pytorch-output',
    sagemaker_session=session,
    distribution={
        'torch_distributed': {
            'enabled': True
        }
    }
)

# Train with distributed strategy
pytorch_estimator.fit(
    {'training': f's3://{bucket}/train-data/'},
    job_name='distributed-training'
)

## Training Job Configuration

```json
{
  "training_job_config": {
    "job_name": "my-training-job",
    "role_arn": "arn:aws:iam::123456789012:role/SageMakerRole",
    "algorithm_specification": {
      "training_image": "382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest",
      "training_input_mode": "File"
    },
    "input_data_config": [
      {
        "channel_name": "training",
        "data_source": {
          "s3_data_source": {
            "s3_data_type": "S3Prefix",
            "s3_uri": "s3://my-bucket/train-data/",
            "s3_data_distribution_type": "FullyReplicated"
          }
        }
      }
    ],
    "output_data_config": {
      "s3_output_path": "s3://my-bucket/training-output/"
    },
    "resource_config": {
      "instance_type": "ml.m5.xlarge",
      "instance_count": 1,
      "volume_size_in_gb": 30
    },
    "stopping_condition": {
      "max_runtime_in_seconds": 86400
    }
  }
}
```

## Monitoring Training Progress

In [ ]:
# Check training job status
import boto3

sm_client = boto3.client('sagemaker')

response = sm_client.describe_training_job(
    TrainingJobName='my-training-job'
)

print(f"Status: {response['TrainingJobStatus']}")
print(f"Start time: {response['CreationTime']}")
print(f"Training time: {response.get('TrainingEndTime', 'In progress')}")
print(f"Billable seconds: {response['BillableTimeInSeconds']}")

## Quiz 1

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary purpose of SageMaker Training Jobs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>Data storage</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>Train ML models on managed infrastructure</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>Real-time predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>Model monitoring</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ How much can spot training save on compute costs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>Up to 90%</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>Up to 30%</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>Up to 50%</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>No savings</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is required for distributed training?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Only one instance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Custom code only</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Multiple instances and distribution configuration</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>GPU instances only</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does the Estimator API provide?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>Only data storage</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>High-level interface for training and deployment</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Only monitoring</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Only hyperparameter tuning</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ Where does SageMaker store training output?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>S3</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>Local filesystem</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>DynamoDB</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>RDS</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>